# AdaptAuth-ML · SuspiciousLogin + CERT r4.2 (Google Colab)

**Desde el celular:** **Entorno de ejecución → Ejecutar todo** y acepta el permiso de Google Drive.

| Parte | Dataset | Qué valida | Tiempo |
|---|---|---|---|
| A | SuspiciousLogin (Google Workspace) | Señales de identidad y red: IP nueva, país, viaje imposible, hora | 2 a 3 min |
| B | CERT r4.2 (CMU) | Historial del empleado: horario, PC habitual, PC de otra persona | Primera vez 25 a 40 min (descarga 4.8 GB); después 3 a 5 min |

Los resultados se guardan en tu Drive: **Mi unidad → AdaptAuth-ML → results_sl** y **results_cert**.

## 0. Código

In [ ]:
%%writefile adaptauth.py
"""
AdaptAuth-ML - pipeline del baseline S8 (Isolation Forest + reglas contextuales).

Implementa el diseño corregido del PI7 (Fase 0):
  - x_i en R^8 (hora cíclica, desviación horaria, historial, frecuencias de país/ASN/dispositivo, RTT)
  - c_i = 0.25 r1 + 0.25 r2 + 0.30 r3 + 0.15 r4 + 0.05 r5
  - s(x) de Isolation Forest (ecuación 1) y reescalado s'(x) (ecuación 1b)
  - R_i = alpha * s'(x_i) + (1 - alpha) * c_i (ecuación 2)
  - Política: acceso directo / OTP / bloqueo, con regla de arranque en frío (P5)
  - Partición temporal 70/15/15 y calibración de alpha, theta1, theta2 solo en validación

Los nombres de columnas siguen el dataset RBA (Wiefling et al., 2022), de modo que el
mismo código corre sobre el archivo real o sobre los datos sintéticos de prueba.
"""
from __future__ import annotations

import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest

# --------------------------------------------------------------------------------------
# Columnas del dataset RBA
# --------------------------------------------------------------------------------------
COL_TS = "Login Timestamp"
COL_USER = "User ID"
COL_RTT = "Round-Trip Time [ms]"
COL_IP = "IP Address"
COL_COUNTRY = "Country"
COL_ASN = "ASN"
COL_BROWSER = "Browser Name and Version"
COL_OS = "OS Name and Version"
COL_DEVICE = "Device Type"
COL_SUCCESS = "Login Successful"
COL_ATTACK_IP = "Is Attack IP"          # SOLO etiqueta de evaluación, nunca entrada
COL_ATO = "Is Account Takeover"         # SOLO etiqueta de evaluación, nunca entrada

LABEL_COLUMNS = (COL_ATTACK_IP, COL_ATO)

FEATURES = [
    "hora_sin", "hora_cos", "desv_horaria", "log_n_prev",
    "frec_pais", "frec_asn", "frec_dispositivo", "log_rtt",
]
# Contexto global (v4): no usa etiquetas, solo el pasado de TODA la población
FEATURES_GLOBAL = ["es_movil", "es_desktop", "tasa_fallo_asn", "tasa_fallo_pais", "log_vol_asn"]
# v5: se quitan variables que crecen con el tiempo (log_vol_asn, log_rtt casi sin datos) y se agregan
# señales por IP, por user agent y por fallos de la cuenta
FEATURES_V5 = ["hora_sin", "hora_cos", "desv_horaria", "log_n_prev", "frec_pais", "frec_asn",
               "frec_dispositivo", "es_movil", "es_desktop", "tasa_fallo_asn", "tasa_fallo_pais",
               "tasa_fallo_ua", "ip_tasa_fallo", "ip_log_intentos_prev", "ip_log_usuarios_prev",
               "ip_log_horas_desde_prev", "ip_nueva_para_usuario", "usr_tasa_fallo",
               "usr_fallos_recientes", "usr_log_horas_desde_prev"]
RULES = ["r1_dispositivo_nuevo", "r2_pais_poco_habitual", "r3_viaje_imposible",
         "r4_horario_atipico", "r5_asn_nuevo"]
RULE_WEIGHTS = {"r1_dispositivo_nuevo": 0.25, "r2_pais_poco_habitual": 0.25,
                "r3_viaje_imposible": 0.30, "r4_horario_atipico": 0.15, "r5_asn_nuevo": 0.05}

DECISION_NAMES = {0: "acceso_directo", 1: "otp", 2: "bloqueo"}


@dataclass
class Config:
    seed: int = 42
    n_estimators: int = 100          # t
    max_samples: int = 256           # n
    split: tuple = (0.70, 0.15, 0.15)
    alpha_grid: tuple = (0.3, 0.5, 0.7, 0.85, 1.0)
    theta_grid: tuple = tuple(np.round(np.arange(0.02, 0.951, 0.01), 2))
    max_fpr_theta1: float = 0.05     # RNF02 / CA-C2-02
    max_fpr_theta2: float = 0.01     # bloqueo: tolerancia más estricta
    cold_start_otp: bool = True      # regla P5
    n_test_blocks: int = 10          # bloques temporales para la prueba pareada
    label: str = "anomalo"           # columna de verdad usada para calibrar


# --------------------------------------------------------------------------------------
# 1. Datos
# --------------------------------------------------------------------------------------
def _open_csv(path: str):
    if str(path).endswith(".zip"):          # se lee directo del .zip, sin descomprimir ~9 GB
        import zipfile
        zf = zipfile.ZipFile(path)
        member = next(n for n in zf.namelist() if n.endswith(".csv"))
        return zf.open(member)
    return path


def load_rba(path: str, nrows: int | None = None, user_pct: float | None = None,
             chunksize: int = 2_000_000) -> pd.DataFrame:
    """
    Carga el CSV real de RBA (rba-dataset.csv o el .zip de Zenodo).
      - nrows: primeras N filas (ventana de tiempo corta: muchos usuarios sin historial).
      - user_pct: muestreo POR USUARIO sobre todo el archivo (p. ej. 5 = 5 % de usuarios con
        TODOS sus accesos). Conserva historiales completos; es la opción recomendada.
    """
    usecols = [COL_TS, COL_USER, COL_RTT, COL_IP, COL_COUNTRY, COL_ASN, COL_BROWSER,
               COL_OS, COL_DEVICE, COL_SUCCESS, COL_ATTACK_IP, COL_ATO]
    if str(path).endswith(".parquet"):      # muestra ya filtrada y guardada (lectura en segundos)
        df = pd.read_parquet(path)
        df[COL_TS] = pd.to_datetime(df[COL_TS])
        return df
    if user_pct:
        parts = []
        for ch in pd.read_csv(_open_csv(path), usecols=usecols, chunksize=chunksize):
            h = pd.util.hash_pandas_object(ch[COL_USER], index=False).to_numpy() % 10000
            parts.append(ch[h < user_pct * 100])
        df = pd.concat(parts, ignore_index=True)
    else:
        df = pd.read_csv(_open_csv(path), usecols=usecols, nrows=nrows)
    df[COL_TS] = pd.to_datetime(df[COL_TS])
    for col in (COL_SUCCESS, COL_ATTACK_IP, COL_ATO):
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1"])
    return df


CTX_COLS = ["tasa_fallo_asn", "tasa_fallo_pais", "tasa_fallo_ua", "ip_tasa_fallo",
            "ip_log_intentos_prev", "ip_log_usuarios_prev", "ip_log_horas_desde_prev"]


def build_full_context_sample(path: str, user_pct: float, chunksize: int = 2_000_000,
                              m: float = 20.0, verbose: bool = True) -> pd.DataFrame:
    """
    Recorre TODO el archivo en orden temporal y calcula el contexto de red (IP, ASN, país,
    user agent) con el tráfico completo, como lo vería el sistema en producción. Devuelve solo
    los accesos del user_pct % de usuarios, con columnas ctx_* listas para reemplazar las
    calculadas dentro de la muestra. Todo es causal: cada evento solo ve eventos anteriores.
    """
    usecols = [COL_TS, COL_USER, COL_RTT, COL_IP, COL_COUNTRY, COL_ASN, COL_BROWSER,
               COL_OS, COL_DEVICE, COL_SUCCESS, COL_ATTACK_IP, COL_ATO]
    empty = lambda: pd.Series(dtype="float64")
    state = {k: (empty(), empty()) for k in ("asn", "pais", "ua", "ip")}
    ip_users, ip_last = empty(), empty()
    seen = np.empty(0, dtype=np.uint64)
    glob_n = glob_f = 0.0
    t_ref, parts, fila0 = None, [], 0
    t0 = time.time()
    for i, ch in enumerate(pd.read_csv(_open_csv(path), usecols=usecols, chunksize=chunksize)):
        ch = ch.reset_index(drop=True)
        ch["_fila"] = np.arange(fila0, fila0 + len(ch)); fila0 += len(ch)
        ts = pd.to_datetime(ch[COL_TS])
        t_ref = ts.min() if t_ref is None else t_ref
        horas = pd.Series((ts - t_ref).dt.total_seconds().to_numpy() / 3600.0)
        ok = ch[COL_SUCCESS]
        ok = ok.astype(str).str.lower().isin(["true", "1"]) if ok.dtype == object else ok.astype(bool)
        fail = pd.Series((~ok.to_numpy()).astype(int))
        keys = {"asn": ch[COL_ASN].astype(str), "pais": ch[COL_COUNTRY].astype(str),
                "ua": ch[COL_BROWSER].astype(str) + "|" + ch[COL_OS].astype(str) + "|" + ch[COL_DEVICE].astype(str),
                "ip": ch[COL_IP].astype(str)}
        f_cum = fail.cumsum().to_numpy() - fail.to_numpy()
        p0 = (glob_f + f_cum + 1.0) / (glob_n + np.arange(len(ch)) + 4.0)
        prior = {}
        for k, key in keys.items():
            cnt, fl = state[k]
            n_k = key.map(cnt).fillna(0).to_numpy() + fail.groupby(key).cumcount().to_numpy()
            f_k = key.map(fl).fillna(0).to_numpy() + (fail.groupby(key).cumsum() - fail).to_numpy()
            prior[k] = (n_k, f_k)
            state[k] = (cnt.add(key.value_counts(), fill_value=0), fl.add(fail.groupby(key).sum(), fill_value=0))
        out = {}
        for k, name in (("asn", "tasa_fallo_asn"), ("pais", "tasa_fallo_pais"), ("ua", "tasa_fallo_ua")):
            n_k, f_k = prior[k]
            out[name] = (f_k + m * p0) / (n_k + m)
        n_ip, f_ip = prior["ip"]
        out["ip_tasa_fallo"] = (f_ip + 5.0 * out["tasa_fallo_asn"]) / (n_ip + 5.0)
        out["ip_log_intentos_prev"] = np.log1p(n_ip)
        h = pd.util.hash_pandas_object(ch[[COL_IP, COL_USER]].astype(str), index=False).to_numpy()
        first = pd.Series(((~pd.Series(h).duplicated()).to_numpy() & ~np.isin(h, seen)).astype(int))
        users_prev = keys["ip"].map(ip_users).fillna(0).to_numpy() + (first.groupby(keys["ip"]).cumsum() - first).to_numpy()
        out["ip_log_usuarios_prev"] = np.log1p(users_prev)
        ip_users = ip_users.add(first.groupby(keys["ip"]).sum(), fill_value=0)
        seen = np.union1d(seen, h[first.to_numpy() == 1])
        prev = horas.groupby(keys["ip"]).shift(1).fillna(keys["ip"].map(ip_last))
        out["ip_log_horas_desde_prev"] = np.log1p((horas - prev).fillna(24 * 365).clip(lower=0)).to_numpy()
        last = horas.groupby(keys["ip"]).max()
        ip_last = pd.concat([ip_last, last]); ip_last = ip_last[~ip_last.index.duplicated(keep="last")]
        glob_n += len(ch); glob_f += float(fail.sum())
        keep = (pd.util.hash_pandas_object(ch[COL_USER], index=False).to_numpy() % 10000) < user_pct * 100
        sub = ch.loc[keep].copy()
        for name, arr in out.items():
            sub["ctx_" + name] = np.asarray(arr)[keep]
        parts.append(sub)
        if verbose:
            print(f"  bloque {i + 1}: {fila0:,} intentos leídos | IPs vistas {len(ip_users):,} | {time.time() - t0:.0f}s", flush=True)
    df = pd.concat(parts, ignore_index=True)
    df[COL_TS] = pd.to_datetime(df[COL_TS])
    for col in (COL_SUCCESS, COL_ATTACK_IP, COL_ATO):
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1"])
    return df


def apply_full_context(d: pd.DataFrame) -> bool:
    """Si la muestra trae columnas ctx_* (tráfico completo), reemplaza las calculadas en la muestra."""
    if not all(("ctx_" + c) in d.columns for c in CTX_COLS):
        return False
    for c in CTX_COLS:
        d[c] = d["ctx_" + c].to_numpy()
    return True


def generate_synthetic_rba(n_users: int = 2000, days: int = 365, seed: int = 42) -> pd.DataFrame:
    """
    Datos SINTÉTICOS con la misma estructura de columnas que RBA. Sirven solo para
    probar el pipeline de punta a punta; sus métricas NO son resultados de la tesis.
    """
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2020-02-03")
    countries = np.array(["NO", "SE", "DE", "GB", "US", "FR", "ES", "PL", "BR", "IN"])
    home_p = np.array([0.55, 0.08, 0.07, 0.06, 0.06, 0.05, 0.04, 0.04, 0.03, 0.02])
    asn_pool = {c: [10000 + 100 * i + k for k in range(6)] for i, c in enumerate(countries)}
    attacker_countries = np.array(["US", "RU", "CN", "VN", "BR", "NL", "UA", "ID"])
    attacker_asns = [64000 + k for k in range(60)]       # proveedores de hosting
    browsers = ["Chrome 88", "Firefox 85", "Safari 14", "Edge 88", "Chrome Mobile 88",
                "Samsung Internet 13", "Opera 74"]
    oses = ["Windows 10", "Mac OS X 10.15", "iOS 14.4", "Android 10", "Linux", "Android 11"]
    devices = ["desktop", "mobile", "tablet"]
    base_rtt = {c: v for c, v in zip(countries, [30, 45, 60, 65, 140, 70, 90, 70, 220, 260])}

    rows = []
    for uid in range(n_users):
        home = rng.choice(countries, p=home_p)
        user_asns = rng.choice(asn_pool[home], size=rng.integers(1, 3), replace=False)
        user_ips = [f"10.{uid % 250}.{k}.{rng.integers(255)}" for k in range(rng.integers(1, 4))]
        n_dev = rng.integers(1, 4)
        user_devs = [(rng.choice(browsers), rng.choice(oses), rng.choice(devices, p=[.5, .42, .08]))
                     for _ in range(n_dev)]
        pref_hour = np.clip(rng.normal(13, 3), 6, 22)
        n_events = int(np.clip(rng.lognormal(4.2, 0.9), 2, 900))
        times = np.sort(rng.uniform(0, days, n_events))
        for t in times:
            day = np.floor(t)
            hour = (rng.normal(pref_hour, 2.5)) % 24
            ts = start + pd.Timedelta(days=float(day)) + pd.Timedelta(hours=float(hour))
            ctry, asn = home, rng.choice(user_asns)
            if rng.random() < 0.02:                       # viaje legítimo
                ctry = rng.choice(countries)
                asn = rng.choice(asn_pool[ctry])
            elif rng.random() < 0.08:                     # red móvil / otra red del país
                asn = rng.choice(asn_pool[home])
            dev = user_devs[rng.integers(len(user_devs))]
            if rng.random() < 0.03:                       # dispositivo nuevo legítimo
                dev = (rng.choice(browsers), rng.choice(oses), rng.choice(devices))
                user_devs.append(dev)
            rtt = rng.lognormal(np.log(base_rtt[ctry]), 0.35)
            ip = (f"100.64.{rng.integers(40)}.{rng.integers(255)}" if rng.random() < 0.15   # NAT móvil compartida
                  else user_ips[rng.integers(len(user_ips))])
            rows.append((ts, uid, rtt, ctry, asn, dev[0], dev[1], dev[2],
                         rng.random() > 0.04, 0, 0, ip))

    n_legit = len(rows)
    # Relleno de credenciales (credential stuffing) desde IPs de ataque
    n_attack = int(n_legit * 0.08)
    for _ in range(n_attack):
        uid = int(rng.integers(n_users))
        ts = start + pd.Timedelta(days=float(rng.uniform(0, days)))
        ctry = rng.choice(attacker_countries)
        rtt = rng.lognormal(np.log(180), 0.6)
        rows.append((ts, uid, rtt, ctry, rng.choice(attacker_asns), rng.choice(browsers),
                     rng.choice(oses), "desktop", rng.random() < 0.02, 1, 0,
                     f"203.0.{rng.integers(150)}.1"))
    # Toma de cuenta (account takeover): acceso exitoso del atacante
    n_ato = int(n_legit * 0.002)
    for _ in range(n_ato):
        uid = int(rng.integers(n_users))
        ts = start + pd.Timedelta(days=float(rng.uniform(30, days)))
        ctry = rng.choice(attacker_countries)
        rtt = rng.lognormal(np.log(180), 0.6)
        rows.append((ts, uid, rtt, ctry, rng.choice(attacker_asns), rng.choice(browsers),
                     rng.choice(oses), rng.choice(devices), True, 1, 1, f"203.0.{rng.integers(150)}.1"))

    df = pd.DataFrame(rows, columns=[COL_TS, COL_USER, COL_RTT, COL_COUNTRY, COL_ASN,
                                     COL_BROWSER, COL_OS, COL_DEVICE, COL_SUCCESS,
                                     COL_ATTACK_IP, COL_ATO, COL_IP])
    return df


# --------------------------------------------------------------------------------------
# 2. C1 - Ingesta y feature engineering (solo usa el pasado de cada usuario)
# --------------------------------------------------------------------------------------
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construye x_i (R^8), las reglas r1..r5 y c_i. El historial de cada usuario se forma
    SOLO con sus inicios de sesión exitosos anteriores al evento, de modo que no hay
    fuga temporal ni se usan las etiquetas de ataque como entrada.
    """
    d = df.sort_values(COL_TS, kind="mergesort").reset_index(drop=True).copy()
    d["anomalo"] = ((d[COL_ATTACK_IP] == 1) | (d[COL_ATO] == 1)).astype(int)
    d["dispositivo"] = (d[COL_BROWSER].astype(str) + "|" + d[COL_OS].astype(str) + "|"
                        + d[COL_DEVICE].astype(str))
    succ = d[COL_SUCCESS].astype(bool).astype(int)
    d["_succ"] = succ

    hour = d[COL_TS].dt.hour + d[COL_TS].dt.minute / 60.0
    ang = 2 * np.pi * hour / 24.0
    d["hora_sin"], d["hora_cos"] = np.sin(ang), np.cos(ang)

    g = d.groupby(COL_USER)
    n_prev = g["_succ"].cumsum() - succ                   # accesos exitosos previos
    d["n_prev"] = n_prev
    d["log_n_prev"] = np.log1p(n_prev)

    def prior_share(key: str) -> pd.Series:
        same = d.groupby([COL_USER, key])["_succ"].cumsum() - succ
        return np.where(n_prev > 0, same / n_prev.clip(lower=1), 0.0)

    d["frec_pais"] = prior_share(COL_COUNTRY)
    d["frec_asn"] = prior_share(COL_ASN)
    d["frec_dispositivo"] = prior_share("dispositivo")

    # Hora habitual: media circular de los accesos exitosos previos
    s_cum = (d["hora_sin"] * succ).groupby(d[COL_USER]).cumsum() - d["hora_sin"] * succ
    c_cum = (d["hora_cos"] * succ).groupby(d[COL_USER]).cumsum() - d["hora_cos"] * succ
    mean_ang = np.arctan2(s_cum, c_cum)
    diff = np.abs(np.angle(np.exp(1j * (ang - mean_ang))))     # 0..pi
    desv_h = diff * 24 / (2 * np.pi)                            # 0..12 h
    d["desv_horas"] = np.where(n_prev > 0, desv_h, np.nan)
    d["desv_horaria"] = np.where(n_prev > 0, desv_h / 12.0, 0.5)

    d["log_rtt"] = np.log1p(d[COL_RTT].astype(float).fillna(d[COL_RTT].median()))

    # Último acceso exitoso previo (para viaje imposible)
    d["_ctry_s"] = d[COL_COUNTRY].where(succ == 1)
    d["_ts_s"] = d[COL_TS].where(succ == 1)
    # Vectorizado (sin lambda por usuario) para soportar millones de filas
    d["prev_pais"] = d.groupby(COL_USER)["_ctry_s"].shift(1).groupby(d[COL_USER]).ffill()
    d["prev_ts"] = d.groupby(COL_USER)["_ts_s"].shift(1).groupby(d[COL_USER]).ffill()
    horas_desde_prev = (d[COL_TS] - d["prev_ts"]).dt.total_seconds() / 3600.0

    hist = n_prev >= 1
    d["r1_dispositivo_nuevo"] = (hist & (d["frec_dispositivo"] == 0)).astype(int)
    d["r2_pais_poco_habitual"] = (hist & (d["frec_pais"] < 0.2)).astype(int)
    d["r3_viaje_imposible"] = (d["prev_pais"].notna() & (d["prev_pais"] != d[COL_COUNTRY])
                               & (horas_desde_prev < 2)).astype(int)
    d["r4_horario_atipico"] = ((n_prev >= 5) & (d["desv_horas"] > 4)).astype(int)
    d["r5_asn_nuevo"] = (hist & (d["frec_asn"] == 0)).astype(int)
    d["c_i"] = sum(RULE_WEIGHTS[r] * d[r] for r in RULES)
    d["arranque_frio"] = (n_prev == 0).astype(int)

    # ---- Contexto global (v4), causal: cada evento solo ve eventos anteriores ----
    dev = d[COL_DEVICE].astype(str).str.lower()
    d["es_movil"] = (dev == "mobile").astype(int)
    d["es_desktop"] = (dev == "desktop").astype(int)
    fail = 1 - succ
    g_n = np.arange(len(d))
    g_f = fail.cumsum().to_numpy() - fail.to_numpy()
    p0 = (g_f + 1.0) / (g_n + 4.0)                      # tasa global de fallo previa (suavizada)
    m = 20.0                                            # fuerza del suavizado bayesiano
    for key, name in ((COL_ASN, "tasa_fallo_asn"), (COL_COUNTRY, "tasa_fallo_pais")):
        n_k = d.groupby(key).cumcount().to_numpy()
        f_k = fail.groupby(d[key]).cumsum().to_numpy() - fail.to_numpy()
        d[name] = (f_k + m * p0) / (n_k + m)
    d["log_vol_asn"] = np.log1p(d.groupby(COL_ASN).cumcount())

    # ---- v5: IP, user agent y fallos de la cuenta (causales, sin etiquetas) ----
    horas = (d[COL_TS] - d[COL_TS].min()).dt.total_seconds() / 3600.0
    ip = d[COL_IP].astype(str)
    ip_n = d.groupby(ip).cumcount()
    ip_f = fail.groupby(ip).cumsum() - fail
    d["ip_tasa_fallo"] = (ip_f + 5.0 * d["tasa_fallo_asn"]) / (ip_n + 5.0)   # previa jerárquica: ASN
    d["ip_log_intentos_prev"] = np.log1p(ip_n)
    first_pair = (~d.duplicated([COL_IP, COL_USER])).astype(int)
    d["ip_log_usuarios_prev"] = np.log1p(first_pair.groupby(ip).cumsum() - first_pair)
    d["ip_log_horas_desde_prev"] = np.log1p(horas.groupby(ip).diff().fillna(24 * 365).clip(lower=0))
    pair_ok_prev = d.groupby([COL_USER, COL_IP])["_succ"].cumsum() - succ
    d["ip_nueva_para_usuario"] = (pair_ok_prev == 0).astype(int)
    ua = d["dispositivo"]
    ua_n = d.groupby(ua).cumcount()
    ua_f = fail.groupby(ua).cumsum() - fail
    d["tasa_fallo_ua"] = (ua_f + m * p0) / (ua_n + m)
    u_n = d.groupby(COL_USER).cumcount()
    u_f = fail.groupby(d[COL_USER]).cumsum() - fail                     # fallos previos de la cuenta
    d["usr_tasa_fallo"] = (u_f + 3.0 * p0) / (u_n + 3.0)
    d["usr_fallos_recientes"] = u_f - u_f.groupby(d[COL_USER]).shift(5).fillna(0)   # en sus 5 intentos previos
    d["usr_log_horas_desde_prev"] = np.log1p(horas.groupby(d[COL_USER]).diff().fillna(24 * 365).clip(lower=0))

    return d.drop(columns=["_succ", "_ctry_s", "_ts_s"])


def diagnose(raw: pd.DataFrame, d: pd.DataFrame) -> dict:
    """
    Análisis exploratorio orientado a la detección: ¿qué variables separan de verdad los
    accesos anómalos de los legítimos? (Etapa 1 del procedimiento metodológico).
    """
    from sklearn.metrics import roc_auc_score
    y = d["anomalo"].to_numpy()
    lab = np.where(y == 1, "anomalo", "legitimo")
    per_user = d.groupby(COL_USER).size()

    def auc(col, mask=None):
        m = np.ones(len(d), bool) if mask is None else mask
        yy, xx = y[m], d.loc[m, col].astype(float).fillna(0).to_numpy()
        return round(float(roc_auc_score(yy, xx)), 3) if 0 < yy.sum() < len(yy) else None

    hist = (d["n_prev"] >= 1).to_numpy()
    cols = FEATURES + FEATURES_GLOBAL + RULES + ["c_i", COL_RTT]

    def top_share(col, k=5):
        out = {}
        for name in ("legitimo", "anomalo"):
            s = d.loc[lab == name, col].astype(str).value_counts(normalize=True).head(k)
            out[name] = {i: round(float(v), 3) for i, v in s.items()}
        return out

    legit_asn = set(d.loc[y == 0, COL_ASN])
    legit_users = set(d.loc[(y == 0) & d[COL_SUCCESS].astype(bool), COL_USER])
    atk = d[y == 1]
    return {
        "archivo_ordenado_por_fecha": bool(raw[COL_TS].is_monotonic_increasing),
        "rango_fechas": [str(d[COL_TS].min()), str(d[COL_TS].max())],
        "eventos": int(len(d)), "usuarios": int(per_user.size),
        "eventos_por_usuario_p50_p90_max": [int(per_user.quantile(.5)), int(per_user.quantile(.9)),
                                            int(per_user.max())],
        "tasa_anomalos": round(float(y.mean()), 4), "n_ato": int(d[COL_ATO].sum()),
        "exito_login": {k: round(float(d.loc[lab == k, COL_SUCCESS].astype(bool).mean()), 4)
                        for k in ("legitimo", "anomalo")},
        "primer_acceso": {k: round(float((d.loc[lab == k, "n_prev"] == 0).mean()), 4)
                          for k in ("legitimo", "anomalo")},
        "anomalos_sobre_usuarios_con_historial_legitimo": round(float(atk[COL_USER].isin(legit_users).mean()), 4),
        "anomalos_con_asn_visto_en_legitimos": round(float(atk[COL_ASN].isin(legit_asn).mean()), 4),
        "pais_top5": top_share(COL_COUNTRY), "tipo_dispositivo_top5": top_share(COL_DEVICE),
        "asn_top5": top_share(COL_ASN),
        "auc_univariada_todos": {c: auc(c) for c in cols},
        "auc_univariada_con_historial": {c: auc(c, hist) for c in cols},
        "tasa_reglas": {r: {k: round(float(d.loc[lab == k, r].mean()), 4) for k in ("legitimo", "anomalo")}
                        for r in RULES},
        "nota_auc": "0.5 = no separa; > 0.5 = valores altos indican anomalía; < 0.5 = relación invertida",
    }


def temporal_split(d: pd.DataFrame, split=(0.70, 0.15, 0.15)):
    n = len(d)
    a, b = int(n * split[0]), int(n * (split[0] + split[1]))
    return d.iloc[:a].copy(), d.iloc[a:b].copy(), d.iloc[b:].copy()


# --------------------------------------------------------------------------------------
# 3. C2 / C3 / C4 - detección, consolidación y decisión
# --------------------------------------------------------------------------------------
def train_iforest(train: pd.DataFrame, cfg: Config, features=None) -> IsolationForest:
    features = features or FEATURES
    model = IsolationForest(n_estimators=cfg.n_estimators, max_samples=cfg.max_samples,
                            contamination="auto", random_state=cfg.seed, n_jobs=-1)
    model.fit(train[features].to_numpy())
    return model


def anomaly_score(model: IsolationForest, d: pd.DataFrame, features=None) -> np.ndarray:
    """Ecuación 1: s(x, n) = 2^(-E[h(x)]/c(n)). En scikit-learn, score_samples = -s(x, n)."""
    return -model.score_samples(d[features or FEATURES].to_numpy())


def rescale(s: np.ndarray, s_ref: float) -> np.ndarray:
    """Ecuación 1b: s'(x) = max(0, (s - s_ref) / (1 - s_ref))."""
    return np.clip((s - s_ref) / (1.0 - s_ref), 0.0, 1.0)


def risk_score(s_prime: np.ndarray, c: np.ndarray, alpha: float) -> np.ndarray:
    """Ecuación 2: R_i = alpha * s'(x_i) + (1 - alpha) * c_i."""
    return alpha * s_prime + (1.0 - alpha) * c


def decide(R: np.ndarray, theta1: float, theta2: float, cold: np.ndarray | None = None,
           cold_start_otp: bool = True) -> np.ndarray:
    """Reglas P2, P3 y P5. 0 = acceso directo, 1 = OTP, 2 = bloqueo."""
    dec = np.where(R > theta2, 2, np.where(R > theta1, 1, 0))
    if cold_start_otp and cold is not None:
        dec = np.where((cold == 1) & (dec == 0), 1, dec)
    return dec


# --------------------------------------------------------------------------------------
# 4. Métricas (dos análisis binarios sobre la decisión de tres niveles)
# --------------------------------------------------------------------------------------
def binary_metrics(y: np.ndarray, pred: np.ndarray) -> dict:
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0
    fnr = fn / (fn + tp) if fn + tp else 0.0
    denom = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denom if denom else 0.0
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=precision, recall=recall, f1=f1,
                fpr=fpr, fnr=fnr, mcc=mcc)


def evaluate_decisions(y: np.ndarray, dec: np.ndarray) -> dict:
    """Análisis theta1: positivo = OTP o bloqueo. Análisis theta2: positivo = bloqueo."""
    return {"analisis_theta1": binary_metrics(y, (dec >= 1).astype(int)),
            "analisis_theta2": binary_metrics(y, (dec == 2).astype(int))}


def confusion_3x2(y: np.ndarray, dec: np.ndarray) -> pd.DataFrame:
    tab = pd.crosstab(pd.Series(dec).map(DECISION_NAMES).rename("decision"),
                      pd.Series(y).map({0: "legitimo", 1: "anomalo"}).rename("etiqueta_real"))
    return tab.reindex(index=list(DECISION_NAMES.values()), columns=["legitimo", "anomalo"],
                       fill_value=0)


# --------------------------------------------------------------------------------------
# 5. Calibración en validación (el conjunto de prueba no se usa aquí)
# --------------------------------------------------------------------------------------
def calibrate(val: pd.DataFrame, s_prime: np.ndarray, cfg: Config, alpha_grid) -> dict:
    """
    Búsqueda en malla: maximiza F1 del análisis theta1 sujeto a FPR <= max_fpr_theta1.
    theta2 es el menor umbral > theta1 cuyo FPR de bloqueo <= max_fpr_theta2.
    """
    y = val[cfg.label].to_numpy()
    cold = val["arranque_frio"].to_numpy()
    c = val["c_i"].to_numpy()
    best, fallback = None, None
    for alpha in alpha_grid:
        R = risk_score(s_prime, c, alpha)
        for t1 in cfg.theta_grid:
            dec = decide(R, t1, 1.01, cold, cfg.cold_start_otp)
            m = binary_metrics(y, (dec >= 1).astype(int))
            cand = dict(alpha=float(alpha), theta1=float(t1), f1_val=m["f1"], fpr_val=m["fpr"])
            if m["fpr"] <= cfg.max_fpr_theta1 and (best is None or m["f1"] > best["f1_val"]):
                best = cand
            # respaldo: menor FPR posible y, a igualdad, mayor F1
            if fallback is None or (m["fpr"], -m["f1"]) < (fallback["fpr_val"], -fallback["f1_val"]):
                fallback = cand
    if best is None:   # ninguna combinación cumple FPR <= umbral: se declara y se usa el respaldo
        best = dict(fallback, restriccion_fpr_incumplida=True)
    R = risk_score(s_prime, c, best["alpha"])
    best["theta2"] = 0.95
    for t2 in cfg.theta_grid:
        if t2 <= best["theta1"]:
            continue
        m = binary_metrics(y, (R > t2).astype(int))
        if m["fpr"] <= cfg.max_fpr_theta2:
            best["theta2"] = float(t2)
            break
    return best


# --------------------------------------------------------------------------------------
# 6. Prueba pareada por bloques temporales (baseline vs AdaptAuth-ML)
# --------------------------------------------------------------------------------------
def paired_block_test(test: pd.DataFrame, dec_a: np.ndarray, dec_b: np.ndarray,
                      y_col: str, n_blocks: int) -> dict:
    """
    Divide el conjunto de prueba en n bloques temporales consecutivos y compara el F1
    (análisis theta1) de ambas condiciones sobre los MISMOS bloques: muestras pareadas.
    """
    y = test[y_col].to_numpy()
    idx = np.array_split(np.arange(len(test)), n_blocks)
    f1_a = np.array([binary_metrics(y[i], (dec_a[i] >= 1).astype(int))["f1"] for i in idx])
    f1_b = np.array([binary_metrics(y[i], (dec_b[i] >= 1).astype(int))["f1"] for i in idx])
    diff = f1_b - f1_a
    shapiro_p = float(stats.shapiro(diff).pvalue) if np.ptp(diff) > 0 else 1.0
    if shapiro_p > 0.05:
        t = stats.ttest_rel(f1_b, f1_a)
        test_name, stat, p = "t de Student pareada", float(t.statistic), float(t.pvalue)
    else:
        w = stats.wilcoxon(f1_b, f1_a)
        test_name, stat, p = "Wilcoxon de rangos con signo", float(w.statistic), float(w.pvalue)
    sd = diff.std(ddof=1)
    dz = float(diff.mean() / sd) if sd > 0 else float("inf")
    return dict(n_bloques=n_blocks, f1_baseline=f1_a.tolist(), f1_adaptauth=f1_b.tolist(),
                media_baseline=float(f1_a.mean()), media_adaptauth=float(f1_b.mean()),
                de_baseline=float(f1_a.std(ddof=1)), de_adaptauth=float(f1_b.std(ddof=1)),
                shapiro_p_diferencias=shapiro_p, prueba=test_name, estadistico=stat, p_valor=p,
                d_cohen_dz=dz)


# --------------------------------------------------------------------------------------
# 7. Latencia (RNF03 / CA-C2-03)
# --------------------------------------------------------------------------------------
def inference_latency_ms(model, d: pd.DataFrame, n: int = 300, seed: int = 0, features=None) -> dict:
    rng = np.random.default_rng(seed)
    X = d[features or FEATURES].to_numpy()
    score = model.score_samples if hasattr(model, "score_samples") else model.predict_proba
    times = []
    for i in rng.integers(0, len(X), n):
        t0 = time.perf_counter()
        score(X[i:i + 1])
        times.append((time.perf_counter() - t0) * 1000)
    t = np.array(times)
    return dict(p50_ms=float(np.percentile(t, 50)), p95_ms=float(np.percentile(t, 95)),
                p99_ms=float(np.percentile(t, 99)), media_ms=float(t.mean()))


def to_json(obj, path):
    def conv(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, Config):
            return asdict(o)
        raise TypeError(type(o))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=conv)


In [ ]:
%%writefile figures.py
"""Figuras del experimento base (paleta de referencia: azul y naranja, marcas finas)."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150,
})


def _grid(ax, axis="y"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def risk_distribution(R, y, t1, t2, path):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    bins = np.linspace(0, 1, 41)
    ax.hist(R[y == 0], bins=bins, density=True, histtype="step", linewidth=2, color=BLUE,
            label="Accesos legítimos")
    ax.hist(R[y == 1], bins=bins, density=True, histtype="step", linewidth=2, color=ORANGE,
            label="Accesos anómalos")
    ax.set_yscale("log")
    top = ax.get_ylim()[1]
    for t, name, f, ha, dx in ((t1, "θ1", 0.5, "right", -0.01), (t2, "θ2", 0.12, "left", 0.01)):
        ax.axvline(t, color=INK2, linewidth=1, linestyle="--")
        ax.text(t + dx, top * f, f"{name} = {t:.2f}", color=INK2, fontsize=10, ha=ha)
    ax.set_xlabel("Score de riesgo consolidado R_i")
    ax.set_ylabel("Densidad (escala log)")
    _grid(ax)
    ax.legend(frameon=False, loc="upper right")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def paired_blocks(res, path, etiquetas=("Baseline (solo reglas)", "AdaptAuth-ML")):
    a, b = np.array(res["f1_baseline"]), np.array(res["f1_adaptauth"])
    x = np.arange(1, len(a) + 1)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(x, a, color=BLUE, linewidth=2, marker="o", markersize=7, label=etiquetas[0])
    ax.plot(x, b, color=ORANGE, linewidth=2, marker="o", markersize=7, label=etiquetas[1])
    ax.set_xticks(x)
    ax.set_xlabel("Bloque temporal del conjunto de prueba")
    ax.set_ylabel("F1 (análisis θ1)")
    ax.set_ylim(0, 1)
    _grid(ax)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def shap_bars(mean_abs, path, xlabel="Importancia media |SHAP| sobre el score de anomalía"):
    s = mean_abs.sort_values()
    fig, ax = plt.subplots(figsize=(8, 0.32 * len(s) + 1.4))
    ax.barh(s.index, s.values, color=BLUE, height=0.6)
    for i, v in enumerate(s.values):
        ax.text(v, i, f" {v:.3f}", va="center", color=INK2, fontsize=9)
    ax.set_xlabel(xlabel)
    _grid(ax, "x")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def model_comparison(tabla, path):
    """Barras horizontales agrupadas: AUC-ROC y F1 (θ1) por modelo, misma escala 0 a 1."""
    t = tabla.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(t))
    h = 0.36
    fig, ax = plt.subplots(figsize=(8.5, 0.6 * len(t) + 1.6))
    ax.barh(y + h / 2 + 0.01, t["auc_roc"], height=h, color=BLUE, label="AUC-ROC")
    ax.barh(y - h / 2 - 0.01, t["f1_theta1"], height=h, color=ORANGE, label="F1 (análisis θ1)")
    for i, (a, f) in enumerate(zip(t["auc_roc"], t["f1_theta1"])):
        ax.text(a + 0.01, i + h / 2, f"{a:.2f}", va="center", color=INK2, fontsize=9)
        ax.text(f + 0.01, i - h / 2, f"{f:.2f}", va="center", color=INK2, fontsize=9)
    ax.axvline(0.5, color=INK2, linewidth=1, linestyle="--")
    ax.text(0.505, -0.75, "azar (AUC 0.5)", color=INK2, fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(t["modelo"].str.replace("_", " "))
    ax.set_xlim(0, 1.08)
    ax.set_ylim(-0.9, len(t) - 0.4)
    _grid(ax, "x")
    ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def operating_points(po, path, titulo=""):
    """Recall y precisión en prueba según el FPR fijado en validación (puntos de operación)."""
    fig, ax = plt.subplots(figsize=(8, 4.2))
    x = po["fpr_prueba"] * 100
    ax.plot(x, po["recall"], color=BLUE, linewidth=2, marker="o", markersize=7, label="Recall (ataques detectados)")
    ax.plot(x, po["precision"], color=ORANGE, linewidth=2, marker="o", markersize=7, label="Precisión")
    for xi, f in zip(x, po["f1"]):
        ax.annotate(f"F1 {f:.2f}", (xi, 0.02), color=INK2, fontsize=8, ha="center")
    ax.set_xscale("log")
    ax.set_xticks(x); ax.set_xticklabels([f"{v:.1f}%" for v in x], fontsize=9)
    ax.minorticks_off()
    ax.set_xlabel("FPR en prueba (legítimos a los que se les pide OTP)")
    ax.set_ylim(0, 1.02)
    _grid(ax)
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    if titulo:
        ax.set_title(f"Puntos de operación: {titulo.replace('_', ' ')}", color=INK2, fontsize=11, loc="left")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


In [ ]:
%%writefile cert.py
"""
CERT Insider Threat r4.2 (CMU, CC BY 4.0) para el componente de historial del usuario de AdaptAuth-ML.

Unidad de análisis: usuario-día (cada día de cada empleado con actividad de inicio de sesión).
Solo se usan los registros de inicio/cierre de sesión (logon.csv) y la estructura de la empresa (LDAP):
no se usan USB, web ni correo, para que el modelo siga siendo de AUTENTICACIÓN y no de amenaza interna.

Etiqueta principal: el usuario-día contiene al menos un inicio de sesión listado como malicioso en las
respuestas oficiales (answers). Etiqueta secundaria: cualquier evento malicioso ese día.
"""
from __future__ import annotations

import glob
import os

import numpy as np
import pandas as pd

FMT = "%m/%d/%Y %H:%M:%S"
FEATURES_CERT = ["n_logons", "n_eventos", "primera_hora", "ultima_hora", "n_fuera_horario", "frac_fuera_horario",
                 "n_pcs", "n_pc_nuevo", "n_pc_de_otro", "fin_de_semana", "z_primera_hora", "z_ultima_hora",
                 "z_n_logons", "z_n_pcs", "dias_desde_ultimo", "log_dias_historial"]
REGLAS_CERT = {"r1_pc_de_otro": 0.30, "r2_pc_nuevo": 0.25, "r3_fuera_horario": 0.25,
               "r4_fin_de_semana": 0.10, "r5_inicio_atipico": 0.10}


def buscar(base: str, patron: str) -> list[str]:
    return sorted(glob.glob(os.path.join(base, "**", patron), recursive=True))


def cargar_logon(path: str) -> pd.DataFrame:
    lg = pd.read_csv(path)
    lg.columns = [c.strip().lower() for c in lg.columns]
    lg["date"] = pd.to_datetime(lg["date"], format=FMT, errors="coerce")
    return lg.dropna(subset=["date"])


def cargar_ldap(carpeta: str) -> pd.DataFrame:
    """Último registro conocido de cada empleado (unidad funcional, departamento, cargo)."""
    archivos = buscar(carpeta, "*.csv")
    if not archivos:
        return pd.DataFrame(columns=["user"])
    partes = []
    for f in archivos:
        t = pd.read_csv(f)
        t.columns = [c.strip().lower() for c in t.columns]
        t["_mes"] = os.path.basename(f)
        partes.append(t)
    ld = pd.concat(partes, ignore_index=True).sort_values("_mes")
    ld = ld.rename(columns={"user_id": "user"}).drop_duplicates("user", keep="last")
    cols = [c for c in ("user", "role", "business_unit", "functional_unit", "department", "team", "supervisor")
            if c in ld.columns]
    return ld[cols]


def cargar_respuestas(carpeta: str, release: str = "4.2"):
    """
    Lee answers/insiders.csv y los archivos de detalle del release. Cada línea de detalle empieza con el
    tipo de evento (logon, device, http, email, file) seguido de id, fecha, usuario y PC.
    Devuelve (eventos maliciosos, tabla de insiders).
    """
    ins_path = buscar(carpeta, "insiders.csv")
    if not ins_path:
        raise FileNotFoundError("No se encontró insiders.csv dentro de las respuestas (answers).")
    ins = pd.read_csv(ins_path[0])
    ins.columns = [c.strip().lower() for c in ins.columns]
    ins = ins[ins["dataset"].astype(str).str.replace("r", "", regex=False).str.strip() == release]
    filas = []
    for det in ins["details"].astype(str):
        rutas = buscar(carpeta, det)
        if not rutas:
            continue
        with open(rutas[0], encoding="utf-8", errors="ignore") as fh:
            for linea in fh:
                p = linea.rstrip("\n").split(",", 5)
                if len(p) >= 5:
                    filas.append(p[:5])
    ev = pd.DataFrame(filas, columns=["tipo", "id", "date", "user", "pc"])
    ev["tipo"] = ev["tipo"].str.strip().str.lower()
    ev["date"] = pd.to_datetime(ev["date"], format=FMT, errors="coerce")
    ev["dia"] = ev["date"].dt.normalize()
    return ev, ins


def construir_usuario_dia(lg: pd.DataFrame, ev: pd.DataFrame, fecha_corte_pc: pd.Timestamp) -> pd.DataFrame:
    """Agrega los inicios de sesión por usuario-día con variables causales (solo días previos del usuario)."""
    lg = lg.sort_values("date", kind="mergesort").reset_index(drop=True)
    lg["dia"] = lg["date"].dt.normalize()
    lg["hora"] = lg["date"].dt.hour + lg["date"].dt.minute / 60.0
    es_logon = (lg["activity"].astype(str).str.lower() == "logon")
    lg["es_logon"] = es_logon.astype(int)
    lg["fuera_horario"] = (((lg["hora"] < 7) | (lg["hora"] >= 19)) & es_logon).astype(int)
    lg["pc_nuevo"] = ((~lg.duplicated(["user", "pc"])) & es_logon).astype(int)
    # PC "de otro": su usuario principal (en los primeros meses) es otra persona
    base = lg[(lg["date"] < fecha_corte_pc) & es_logon]
    dueno = (base.groupby(["pc", "user"]).size().reset_index(name="n").sort_values("n")
             .drop_duplicates("pc", keep="last").set_index("pc")["user"])
    lg["pc_dueno"] = lg["pc"].map(dueno)
    lg["pc_de_otro"] = (lg["pc_dueno"].notna() & (lg["pc_dueno"] != lg["user"]) & es_logon).astype(int)
    ids_mal = set(ev.loc[ev["tipo"] == "logon", "id"])
    lg["malicioso"] = lg["id"].isin(ids_mal).astype(int)

    a = lg.groupby(["user", "dia"]).agg(
        n_logons=("es_logon", "sum"), n_eventos=("id", "size"), primera_hora=("hora", "min"),
        ultima_hora=("hora", "max"), n_fuera_horario=("fuera_horario", "sum"), n_pcs=("pc", "nunique"),
        n_pc_nuevo=("pc_nuevo", "sum"), n_pc_de_otro=("pc_de_otro", "sum"), anomalo=("malicioso", "max"),
    ).reset_index().sort_values(["user", "dia"], kind="mergesort").reset_index(drop=True)

    g = a.groupby("user")
    a["dias_historial"] = g.cumcount()
    n = a["dias_historial"].replace(0, np.nan)
    for col in ("primera_hora", "ultima_hora", "n_logons", "n_pcs"):
        x = a[col].astype(float)
        s1 = x.groupby(a["user"]).cumsum() - x
        s2 = (x ** 2).groupby(a["user"]).cumsum() - x ** 2
        media = s1 / n
        sd = np.sqrt((s2 / n - media ** 2).clip(lower=0))
        a[f"z_{col}"] = ((x - media) / (sd + 0.5)).fillna(0.0)
    a["dias_desde_ultimo"] = g["dia"].diff().dt.days.fillna(30).clip(upper=60)
    a["fin_de_semana"] = (a["dia"].dt.dayofweek >= 5).astype(int)
    a["frac_fuera_horario"] = a["n_fuera_horario"] / a["n_logons"].clip(lower=1)
    a["log_dias_historial"] = np.log1p(a["dias_historial"])
    a["historial_disponible"] = (a["dias_historial"] >= 5).astype(int)
    a["arranque_frio"] = 1 - a["historial_disponible"]

    dias_mal = set(zip(ev["user"], ev["dia"]))
    a["anomalo_cualquier"] = [int(k in dias_mal) for k in zip(a["user"], a["dia"])]

    a["r1_pc_de_otro"] = (a["n_pc_de_otro"] > 0).astype(int)
    a["r2_pc_nuevo"] = ((a["n_pc_nuevo"] > 0) & (a["dias_historial"] >= 5)).astype(int)
    a["r3_fuera_horario"] = (a["n_fuera_horario"] > 0).astype(int)
    a["r4_fin_de_semana"] = a["fin_de_semana"]
    a["r5_inicio_atipico"] = (a["z_primera_hora"].abs() > 2).astype(int)
    a["c_i"] = sum(w * a[r] for r, w in REGLAS_CERT.items())
    return a


def particion_por_usuario(a: pd.DataFrame, seed: int = 42, prop=(0.70, 0.15, 0.15)):
    """
    Divide por EMPLEADO (todos sus días en una sola partición), estratificando por si alguna vez tuvo un
    día malicioso. Con tan pocos insiders, una partición temporal dejaría validación o prueba sin casos.
    """
    rng = np.random.default_rng(seed)
    pos = a.groupby("user")["anomalo"].max()
    grupos = {}
    for valor in (0, 1):
        us = pos[pos == valor].index.to_numpy().copy()
        rng.shuffle(us)
        n1, n2 = int(len(us) * prop[0]), int(len(us) * (prop[0] + prop[1]))
        for u in us[:n1]: grupos[u] = "train"
        for u in us[n1:n2]: grupos[u] = "val"
        for u in us[n2:]: grupos[u] = "test"
    part = a["user"].map(grupos)
    return (a[part == "train"].reset_index(drop=True), a[part == "val"].reset_index(drop=True),
            a[part == "test"].reset_index(drop=True))


def generar_cert_sintetico(base: str, n_users: int = 120, dias: int = 200, seed: int = 0):
    """Archivos con el FORMATO de CERT r4.2 (solo para probar el pipeline). No son datos reales."""
    import tarfile
    rng = np.random.default_rng(seed)
    os.makedirs(os.path.join(base, "r4.2", "LDAP"), exist_ok=True)
    users = [f"USR{i:04d}" for i in range(n_users)]
    pcs = {u: f"PC-{1000 + i}" for i, u in enumerate(users)}
    filas, k = [], 0
    inicio = pd.Timestamp("2010-01-04")
    for d in range(dias):
        dia = inicio + pd.Timedelta(days=d)
        if dia.dayofweek >= 5 and rng.random() < 0.9:
            continue
        for u in users:
            if rng.random() < 0.1:
                continue
            h = rng.normal(8.5, 0.6)
            for t, act in ((h, "Logon"), (h + rng.normal(8.5, 0.7), "Logoff")):
                k += 1
                filas.append((f"{{ID-{k:08d}}}", (dia + pd.Timedelta(hours=float(t))).strftime(FMT), u, pcs[u], act))
    malos, det = [], {}
    for j, u in enumerate(users[:6]):
        victima = users[(j + 10) % n_users]
        for d in rng.choice(np.arange(60, dias), size=3, replace=False):
            dia = inicio + pd.Timedelta(days=int(d))
            k += 1
            pc = pcs[victima] if j % 2 else pcs[u]
            usuario = victima if j % 2 else u
            linea = (f"{{ID-{k:08d}}}", (dia + pd.Timedelta(hours=float(rng.uniform(20, 23)))).strftime(FMT), usuario, pc, "Logon")
            filas.append(linea)
            det.setdefault(u, []).append("logon," + ",".join(linea))
    lg = pd.DataFrame(filas, columns=["id", "date", "user", "pc", "activity"])
    lg["_t"] = pd.to_datetime(lg["date"], format=FMT)
    lg.sort_values("_t").drop(columns="_t").to_csv(os.path.join(base, "r4.2", "logon.csv"), index=False)
    ld = pd.DataFrame({"employee_name": users, "user_id": users, "email": [u + "@x.com" for u in users],
                       "role": "Employee", "business_unit": 1,
                       "functional_unit": [f"Unidad {i % 4}" for i in range(n_users)],
                       "department": [f"Depto {i % 8}" for i in range(n_users)], "team": 1, "supervisor": "Jefe"})
    for mes in ("2010-01", "2010-02"):
        ld.to_csv(os.path.join(base, "r4.2", "LDAP", f"{mes}.csv"), index=False)
    os.makedirs(os.path.join(base, "answers", "r4.2-1"), exist_ok=True)
    ins = []
    for u, lineas in det.items():
        nombre = f"r4.2-1-{u}.csv"
        with open(os.path.join(base, "answers", "r4.2-1", nombre), "w") as fh:
            fh.write("\n".join(lineas) + "\n")
        ins.append(dict(dataset=4.2, scenario=1, details=nombre, user=u, start="", end=""))
    pd.DataFrame(ins).to_csv(os.path.join(base, "answers", "insiders.csv"), index=False)
    with tarfile.open(os.path.join(base, "r4.2.tar.bz2"), "w:bz2") as t:
        t.add(os.path.join(base, "r4.2"), arcname="r4.2")
    with tarfile.open(os.path.join(base, "answers.tar.bz2"), "w:bz2") as t:
        t.add(os.path.join(base, "answers"), arcname="answers")


In [ ]:
%%writefile run_suspiciouslogin.py
"""
AdaptAuth-ML sobre SuspiciousLogin Dataset (Google Workspace, 73 528 inicios de sesión exitosos).

Fuente: https://github.com/salomaopena/SuspiciousLoginDataset (CC BY-NC 4.0)
Etiqueta: is_suspicious de Google Workspace (señal operativa de sospecha, no ataque confirmado).

Diseño:
  - C1: las variables ya calculadas por los autores, agrupadas en temporales, red/geo, historial
    del usuario y tipo de autenticación (el archivo público no trae IP, fecha exacta ni dispositivo).
  - Partición temporal por mes: entrenamiento dic-mar, validación abril, prueba mayo-junio
    (julio-agosto excluidos por vacaciones académicas, como hacen los autores).
  - Modelos: reglas estáticas (c_i), Isolation Forest (no supervisado), supervisado
    (HistGradientBoosting) e híbrido R = alpha * s'(IF) + (1 - alpha) * p(supervisado).
  - Calibración en validación: máx. F1 con FPR <= 5 % (θ1, OTP) y FPR de bloqueo <= 1 % (θ2).

Uso:
    python run_suspiciouslogin.py --data ruta/suspicious_logins_public_v1.csv
"""
import argparse
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

sys.path.insert(0, str(Path(__file__).parent))
import adaptauth as aa  # noqa: E402
import figures  # noqa: E402

OUT = Path(__file__).parent / "results_sl"

GRUPOS = {
    "temporal": ["event_hour", "day_of_week", "is_weekend", "is_business_hours", "is_night_login"],
    "red_geo": ["ip_version", "new_ip", "distinct_ips_7d", "distinct_ips_30d", "is_new_country", "is_new_region",
                "is_new_city", "distinct_countries_cumulative", "distinct_regions_cumulative",
                "distinct_cities_cumulative", "country_changed", "continent_changed", "countries_seen_30d",
                "regions_seen_30d", "cities_seen_30d", "geo_jump", "multiple_country_logins_24h",
                "distinct_users_per_network_24h", "travel_distance_km", "travel_speed_kmh", "impossible_travel"],
    "historial": ["logins_24h", "logins_7d", "logins_30d", "avg_logins_per_day", "hours_since_last_login",
                  "days_since_first_login", "abnormal_login_hour", "historial_disponible"],
}
# login_challenge_method se EXCLUYE: es la verificación que Google aplicó porque ya marcó el acceso como
# sospechoso (p. ej. "sms" 100 %, "idv_preregistered_email" 89 % sospechosos). Es una consecuencia de la
# etiqueta, no un dato disponible antes de la decisión: usarla sería fuga de etiqueta.
CATEGORICAS = ["login_type", "continent", "ip_country"]
EXCLUIDAS_POR_FUGA = ["login_challenge_method", "risk_score", "risk_level"]

# Reglas estáticas (Alternativa 1), equivalentes a r1..r5 del PI7 con las variables disponibles
REGLAS = {"new_ip": 0.25, "is_new_country": 0.25, "impossible_travel": 0.30,
          "abnormal_login_hour": 0.15, "geo_cambio_continente": 0.05}


def preparar(d: pd.DataFrame):
    d = d.copy()
    d["historial_disponible"] = (d["hours_since_last_login"] > 0).astype(int)
    d["geo_cambio_continente"] = (d["geo_jump"] == 2).astype(int)
    top_paises = d["ip_country"].value_counts().head(15).index
    d["ip_country"] = d["ip_country"].where(d["ip_country"].isin(top_paises), "Otro")
    num = [c for g in GRUPOS.values() for c in g]
    X = pd.get_dummies(d[num + CATEGORICAS], columns=CATEGORICAS, dummy_na=True, dtype=float)
    X = X.fillna(0.0)
    d["anomalo"] = d["label"].astype(int)
    d["c_i"] = sum(w * d[r].astype(float) for r, w in REGLAS.items())
    d["arranque_frio"] = 1 - d["historial_disponible"]
    grupo_de = {c: g for g, cols in GRUPOS.items() for c in cols}
    grupos_X = {c: grupo_de.get(c, "autenticacion" if c.startswith("login_type") else "red_geo")
                for c in X.columns}
    return d, X, grupos_X


def f1_max(y, s):
    p, r, _ = precision_recall_curve(y, s)
    return float(np.max(np.where(p + r > 0, 2 * p * r / np.maximum(p + r, 1e-12), 0)))


def evaluar(nombre, s_val, s_test, dv, dt, cfg, alpha_grid=(1.0,), c_val=None, c_test=None):
    v = dv.assign(c_i=0.0 if c_val is None else c_val)
    cal = aa.calibrate(v, s_val, cfg, alpha_grid)
    R = aa.risk_score(s_test, np.zeros(len(dt)) if c_test is None else c_test, cal["alpha"])
    R_val = aa.risk_score(s_val, v["c_i"].to_numpy(), cal["alpha"])
    dec = aa.decide(R, cal["theta1"], cal["theta2"], None, False)
    y = dt["anomalo"].to_numpy()
    res = dict(calibracion=cal, **aa.evaluate_decisions(y, dec),
               auc_roc=float(roc_auc_score(y, R)), auc_pr=float(average_precision_score(y, R)),
               f1_max_validacion=f1_max(dv["anomalo"].to_numpy(), R_val))
    for seg, mask in (("con_historial", dt["historial_disponible"].to_numpy() == 1),
                      ("primer_acceso", dt["historial_disponible"].to_numpy() == 0)):
        res[f"theta1_{seg}"] = aa.binary_metrics(y[mask], (dec[mask] >= 1).astype(int))
    m = res["analisis_theta1"]
    print(f"  {nombre:<17} AUC-ROC={res['auc_roc']:.3f} AUC-PR={res['auc_pr']:.3f} | θ1: F1={m['f1']:.3f} "
          f"P={m['precision']:.3f} R={m['recall']:.3f} FPR={m['fpr']:.3%} | θ2: F1={res['analisis_theta2']['f1']:.3f} "
          f"FPR={res['analisis_theta2']['fpr']:.3%} | α={cal['alpha']}"
          + ("  [FPR<=5% no alcanzable]" if cal.get("restriccion_fpr_incumplida") else ""))
    return res, dec, R, R_val


def prueba_pareada_por_grupos(dt, dec_a, dec_b, grupos):
    """Compara el F1 (θ1) de dos condiciones sobre los mismos grupos (mes x día de la semana)."""
    y = dt["anomalo"].to_numpy()
    f_a, f_b = [], []
    for _, idx in dt.groupby(grupos).indices.items():
        f_a.append(aa.binary_metrics(y[idx], (dec_a[idx] >= 1).astype(int))["f1"])
        f_b.append(aa.binary_metrics(y[idx], (dec_b[idx] >= 1).astype(int))["f1"])
    f_a, f_b = np.array(f_a), np.array(f_b)
    diff = f_b - f_a
    sh = float(stats.shapiro(diff).pvalue) if np.ptp(diff) > 0 else 1.0
    if sh > 0.05:
        t = stats.ttest_rel(f_b, f_a); nombre, est, p = "t de Student pareada", float(t.statistic), float(t.pvalue)
    else:
        w = stats.wilcoxon(f_b, f_a); nombre, est, p = "Wilcoxon de rangos con signo", float(w.statistic), float(w.pvalue)
    sd = diff.std(ddof=1)
    return dict(n_grupos=len(diff), f1_baseline=f_a.tolist(), f1_adaptauth=f_b.tolist(),
                media_baseline=float(f_a.mean()), media_adaptauth=float(f_b.mean()),
                de_baseline=float(f_a.std(ddof=1)), de_adaptauth=float(f_b.std(ddof=1)),
                shapiro_p_diferencias=sh, prueba=nombre, estadistico=est, p_valor=p,
                d_cohen_dz=float(diff.mean() / sd) if sd > 0 else None)


def puntos_operacion(R_val, R_test, yv, yt):
    filas = []
    for fpr in (0.005, 0.01, 0.02, 0.03, 0.05, 0.10):
        thr = float(np.quantile(R_val[yv == 0], 1 - fpr))
        m = aa.binary_metrics(yt, (R_test > thr).astype(int))
        filas.append(dict(fpr_objetivo=fpr, umbral=thr, fpr_prueba=m["fpr"], recall=m["recall"],
                          precision=m["precision"], f1=m["f1"]))
    return pd.DataFrame(filas)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    args = ap.parse_args()
    cfg = aa.Config(cold_start_otp=False, label="anomalo")
    OUT.mkdir(exist_ok=True); (OUT / "figuras").mkdir(exist_ok=True)
    t0 = time.time()

    raw = pd.read_csv(args.data)
    d, X, grupos_X = preparar(raw)
    mes = d["month"].replace({12: 0})
    tr, va, te = (mes <= 3).to_numpy(), (mes == 4).to_numpy(), mes.isin([5, 6]).to_numpy()
    dtr, dv, dt = d[tr].reset_index(drop=True), d[va].reset_index(drop=True), d[te].reset_index(drop=True)
    Xtr, Xv, Xt = X[tr].to_numpy(), X[va].to_numpy(), X[te].to_numpy()
    print(f"[datos] {len(d):,} inicios | {d['actor_pseudo_id'].nunique():,} usuarios | {X.shape[1]} variables")
    print(f"        train dic-mar {tr.sum():,} ({dtr.anomalo.mean():.1%}) | val abr {va.sum():,} ({dv.anomalo.mean():.1%})"
          f" | prueba may-jun {te.sum():,} ({dt.anomalo.mean():.1%})")

    res, decs, Rs, Rvs = {}, {}, {}, {}
    print("[modelos] calibrados en validación, evaluados en prueba")
    res["reglas_estaticas"], decs["reglas_estaticas"], Rs["reglas_estaticas"], Rvs["reglas_estaticas"] = evaluar(
        "reglas_estaticas", np.zeros(len(dv)), np.zeros(len(dt)), dv, dt, cfg, (0.0,), dv["c_i"].to_numpy(), dt["c_i"].to_numpy())

    iso = IsolationForest(n_estimators=200, max_samples=256, random_state=cfg.seed, n_jobs=-1).fit(Xtr)
    s_v, s_t = -iso.score_samples(Xv), -iso.score_samples(Xt)
    s_ref = float(np.median(s_v[dv["anomalo"].to_numpy() == 0]))
    sp_v, sp_t = aa.rescale(s_v, s_ref), aa.rescale(s_t, s_ref)
    res["iforest"], decs["iforest"], Rs["iforest"], Rvs["iforest"] = evaluar("iforest", sp_v, sp_t, dv, dt, cfg)
    res["iforest"]["s_ref"] = s_ref

    hgb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0,
                                         random_state=cfg.seed).fit(Xtr, dtr["anomalo"].to_numpy())
    p_v, p_t = hgb.predict_proba(Xv)[:, 1], hgb.predict_proba(Xt)[:, 1]
    res["supervisado"], decs["supervisado"], Rs["supervisado"], Rvs["supervisado"] = evaluar(
        "supervisado", p_v, p_t, dv, dt, cfg)

    res["hibrido"], decs["hibrido"], Rs["hibrido"], Rvs["hibrido"] = evaluar(
        "hibrido", sp_v, sp_t, dv, dt, cfg, (0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0), p_v, p_t)

    yv, yt = dv["anomalo"].to_numpy(), dt["anomalo"].to_numpy()
    tabla = pd.DataFrame([dict(
        modelo=k, auc_roc=r["auc_roc"], auc_pr=r["auc_pr"], f1_theta1=r["analisis_theta1"]["f1"],
        precision_theta1=r["analisis_theta1"]["precision"], recall_theta1=r["analisis_theta1"]["recall"],
        fpr_theta1=r["analisis_theta1"]["fpr"], f1_theta2=r["analisis_theta2"]["f1"], fpr_theta2=r["analisis_theta2"]["fpr"],
        mcc_theta1=r["analisis_theta1"]["mcc"], f1_con_historial=r["theta1_con_historial"]["f1"],
        f1_primer_acceso=r["theta1_primer_acceso"]["f1"], alpha=r["calibracion"]["alpha"],
        theta1=r["calibracion"]["theta1"], theta2=r["calibracion"]["theta2"]) for k, r in res.items()])
    tabla.to_csv(OUT / "tabla_comparativa_sl.csv", index=False)
    po = puntos_operacion(Rvs["hibrido"], Rs["hibrido"], yv, yt)
    po.to_csv(OUT / "puntos_operacion_sl.csv", index=False)
    aa.confusion_3x2(yt, decs["hibrido"]).to_csv(OUT / "matriz_3x2_hibrido_sl.csv")

    # Importancia por grupo de variables (permutando el grupo completo en el modelo supervisado)
    rng = np.random.default_rng(cfg.seed)
    base = average_precision_score(yt, p_t)
    imp_grupo = {}
    cols = list(X.columns)
    for g in sorted(set(grupos_X.values())):
        idx = [i for i, c in enumerate(cols) if grupos_X[c] == g]
        Xp = Xt.copy()
        perm = rng.permutation(len(Xp))
        Xp[:, idx] = Xp[perm][:, idx]
        imp_grupo[g] = float(base - average_precision_score(yt, hgb.predict_proba(Xp)[:, 1]))
    pi = permutation_importance(hgb, Xt, yt, scoring="average_precision", n_repeats=3, random_state=cfg.seed)
    imp_var = pd.Series(pi.importances_mean, index=cols).sort_values(ascending=False)

    lat_if = aa.inference_latency_ms(iso, pd.DataFrame(Xt, columns=cols), features=cols)
    lat_sup = aa.inference_latency_ms(hgb, pd.DataFrame(Xt, columns=cols), features=cols)
    resultados = dict(
        dataset="SuspiciousLogin (Google Workspace, 2 instituciones)", archivo=args.data,
        etiqueta="is_suspicious de Google Workspace (no es ataque confirmado)",
        variables_excluidas_por_fuga=EXCLUIDAS_POR_FUGA,
        particion="train dic-mar | val abr | prueba may-jun (jul-ago excluidos)",
        volumen=dict(total=len(d), usuarios=int(d["actor_pseudo_id"].nunique()), train=int(tr.sum()), val=int(va.sum()),
                     test=int(te.sum()), sospechosos_test=int(yt.sum()),
                     test_con_historial=int(dt["historial_disponible"].sum())),
        modelos=res, puntos_operacion_hibrido=po.round(4).to_dict(orient="records"),
        prueba_pareada_hibrido_vs_reglas=prueba_pareada_por_grupos(dt, decs["reglas_estaticas"], decs["hibrido"], ["month", "day_of_week"]),
        prueba_pareada_hibrido_vs_iforest=prueba_pareada_por_grupos(dt, decs["iforest"], decs["hibrido"], ["month", "day_of_week"]),
        importancia_por_grupo=imp_grupo, importancia_top15=imp_var.head(15).round(4).to_dict(),
        latencia_ms=dict(iforest=lat_if, supervisado=lat_sup, hibrido_p99_aprox=lat_if["p99_ms"] + lat_sup["p99_ms"]),
    )
    aa.to_json(resultados, OUT / "metricas_sl.json")

    c = res["hibrido"]["calibracion"]
    figures.risk_distribution(Rs["hibrido"], yt, c["theta1"], c["theta2"], OUT / "figuras" / "distribucion_riesgo_sl.png")
    figures.model_comparison(tabla, OUT / "figuras" / "comparacion_modelos_sl.png")
    figures.operating_points(po, OUT / "figuras" / "puntos_operacion_sl.png", titulo="AdaptAuth-ML híbrido")
    figures.shap_bars(pd.Series(imp_grupo).clip(lower=0), OUT / "figuras" / "importancia_grupos_sl.png",
                      xlabel="Caída de AUC-PR al permutar el grupo de variables")
    figures.paired_blocks(resultados["prueba_pareada_hibrido_vs_reglas"], OUT / "figuras" / "f1_por_grupo_sl.png",
                          etiquetas=("Reglas estáticas", "AdaptAuth-ML híbrido"))

    print(tabla[["modelo", "auc_roc", "auc_pr", "f1_theta1", "precision_theta1", "recall_theta1", "fpr_theta1",
                 "f1_con_historial", "f1_primer_acceso", "alpha"]].round(3).to_string(index=False))
    print("[puntos de operación, híbrido]"); print(po.round(3).to_string(index=False))
    for k in ("prueba_pareada_hibrido_vs_reglas", "prueba_pareada_hibrido_vs_iforest"):
        p = resultados[k]
        print(f"[{k}] {p['prueba']} ({p['n_grupos']} grupos): est={p['estadistico']:.2f} p={p['p_valor']:.2e} "
              f"dz={p['d_cohen_dz']} | M_A={p['media_baseline']:.3f} M_B={p['media_adaptauth']:.3f}")
    print("[importancia por grupo]", {k: round(v, 4) for k, v in imp_grupo.items()})
    print("[latencia p99 ms] IF", round(lat_if["p99_ms"], 2), "| supervisado", round(lat_sup["p99_ms"], 2))
    print(f"[listo] {time.time() - t0:.0f}s -> {OUT}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_cert.py
"""
AdaptAuth-ML sobre CERT Insider Threat r4.2: componente de historial del empleado (usuario-día).

Uso:
    python run_cert.py --dir carpeta_con_logon_LDAP_y_answers
"""
import argparse
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.insert(0, str(Path(__file__).parent))
import adaptauth as aa  # noqa: E402
import cert  # noqa: E402
import figures  # noqa: E402

OUT = Path(__file__).parent / "results_cert"
F = cert.FEATURES_CERT


def evaluar(nombre, s_val, s_test, dv, dt, cfg, alpha_grid=(1.0,), c_val=None, c_test=None):
    v = dv.assign(c_i=0.0 if c_val is None else c_val)
    cal = aa.calibrate(v, s_val, cfg, alpha_grid)
    R = aa.risk_score(s_test, np.zeros(len(dt)) if c_test is None else c_test, cal["alpha"])
    R_val = aa.risk_score(s_val, v["c_i"].to_numpy(), cal["alpha"])
    dec = aa.decide(R, cal["theta1"], cal["theta2"], None, False)
    y = dt["anomalo"].to_numpy()
    res = dict(calibracion=cal, **aa.evaluate_decisions(y, dec),
               auc_roc=float(roc_auc_score(y, R)) if 0 < y.sum() < len(y) else None,
               auc_pr=float(average_precision_score(y, R)) if y.sum() > 0 else None,
               recall_dias_cualquier_actividad_maliciosa=aa.binary_metrics(dt["anomalo_cualquier"].to_numpy(),
                                                                          (dec >= 1).astype(int))["recall"])
    m = res["analisis_theta1"]
    print(f"  {nombre:<17} AUC-ROC={res['auc_roc'] or 0:.3f} AUC-PR={res['auc_pr'] or 0:.3f} | θ1: F1={m['f1']:.3f} "
          f"P={m['precision']:.3f} R={m['recall']:.3f} FPR={m['fpr']:.3%} | α={cal['alpha']}"
          + ("  [FPR<=5% no alcanzable]" if cal.get("restriccion_fpr_incumplida") else ""))
    return res, dec, R, R_val


def prueba_pareada(dt, dec_a, dec_b, n=10):
    """F1 (θ1) por grupos de empleados de prueba (mismos grupos en ambas condiciones)."""
    y = dt["anomalo"].to_numpy()
    usuarios = np.array(sorted(dt["user"].unique()))
    rng = np.random.default_rng(7); rng.shuffle(usuarios)
    grupo = dt["user"].map({u: i % n for i, u in enumerate(usuarios)}).to_numpy()
    fa, fb = [], []
    for k in range(n):
        idx = grupo == k
        if y[idx].sum() == 0:
            continue
        fa.append(aa.binary_metrics(y[idx], (dec_a[idx] >= 1).astype(int))["f1"])
        fb.append(aa.binary_metrics(y[idx], (dec_b[idx] >= 1).astype(int))["f1"])
    fa, fb = np.array(fa), np.array(fb)
    if len(fa) < 3:
        return dict(nota="muy pocos grupos con casos maliciosos para una prueba pareada", n_grupos=len(fa))
    d = fb - fa
    sh = float(stats.shapiro(d).pvalue) if np.ptp(d) > 0 else 1.0
    t = stats.ttest_rel(fb, fa) if sh > 0.05 else stats.wilcoxon(fb, fa)
    return dict(n_grupos=len(fa), prueba="t de Student pareada" if sh > 0.05 else "Wilcoxon de rangos con signo",
                media_baseline=float(fa.mean()), media_adaptauth=float(fb.mean()), shapiro_p=sh,
                estadistico=float(t.statistic), p_valor=float(t.pvalue),
                d_cohen_dz=float(d.mean() / d.std(ddof=1)) if d.std(ddof=1) > 0 else None,
                f1_baseline=fa.tolist(), f1_adaptauth=fb.tolist())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", required=True, help="carpeta con logon.csv, LDAP/ y answers/")
    args = ap.parse_args()
    cfg = aa.Config(cold_start_otp=False, label="anomalo")
    OUT.mkdir(exist_ok=True); (OUT / "figuras").mkdir(exist_ok=True)
    t0 = time.time()

    logon_path = cert.buscar(args.dir, "logon.csv")[0]
    lg = cert.cargar_logon(logon_path)
    ev, ins = cert.cargar_respuestas(args.dir, "4.2")
    ldap_dirs = [p for p in cert.buscar(args.dir, "LDAP") if Path(p).is_dir()]
    ld = cert.cargar_ldap(ldap_dirs[0]) if ldap_dirs else pd.DataFrame(columns=["user"])
    corte = lg["date"].min() + pd.DateOffset(months=6)
    a = cert.construir_usuario_dia(lg, ev, corte)
    a = a.merge(ld, on="user", how="left")
    print(f"[CERT r4.2] {len(lg):,} eventos de sesión | {lg['user'].nunique():,} empleados | {len(ins)} insiders | "
          f"{len(ev):,} eventos maliciosos ({(ev['tipo'] == 'logon').sum():,} de inicio de sesión)")
    print(f"[usuario-día] {len(a):,} filas | días con inicio de sesión malicioso: {int(a['anomalo'].sum())} | "
          f"días con cualquier actividad maliciosa: {int(a['anomalo_cualquier'].sum())} | {time.time() - t0:.0f}s")
    if a["anomalo"].sum() < 10:
        print("AVISO: muy pocos días con inicio de sesión malicioso; revisa el formato de answers.")

    tr, va, te = cert.particion_por_usuario(a, cfg.seed)
    print(f"[partición por empleado] train {tr['user'].nunique()} ({int(tr.anomalo.sum())} días mal.) | "
          f"val {va['user'].nunique()} ({int(va.anomalo.sum())}) | prueba {te['user'].nunique()} ({int(te.anomalo.sum())})")

    res, decs, Rs, Rvs = {}, {}, {}, {}
    res["reglas_estaticas"], decs["reglas_estaticas"], Rs["reglas_estaticas"], Rvs["reglas_estaticas"] = evaluar(
        "reglas_estaticas", np.zeros(len(va)), np.zeros(len(te)), va, te, cfg, (0.0,), va["c_i"].to_numpy(), te["c_i"].to_numpy())

    iso = IsolationForest(n_estimators=200, max_samples=256, random_state=cfg.seed, n_jobs=-1).fit(tr[F].to_numpy())
    s_v, s_t = -iso.score_samples(va[F].to_numpy()), -iso.score_samples(te[F].to_numpy())
    s_ref = float(np.median(s_v[va["anomalo"].to_numpy() == 0]))
    sp_v, sp_t = aa.rescale(s_v, s_ref), aa.rescale(s_t, s_ref)
    res["iforest"], decs["iforest"], Rs["iforest"], Rvs["iforest"] = evaluar("iforest", sp_v, sp_t, va, te, cfg)

    hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced",
                                         random_state=cfg.seed).fit(tr[F].to_numpy(), tr["anomalo"].to_numpy())
    p_v, p_t = hgb.predict_proba(va[F].to_numpy())[:, 1], hgb.predict_proba(te[F].to_numpy())[:, 1]
    res["supervisado"], decs["supervisado"], Rs["supervisado"], Rvs["supervisado"] = evaluar("supervisado", p_v, p_t, va, te, cfg)
    res["hibrido"], decs["hibrido"], Rs["hibrido"], Rvs["hibrido"] = evaluar(
        "hibrido", sp_v, sp_t, va, te, cfg, (0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0), p_v, p_t)

    tabla = pd.DataFrame([dict(modelo=k, auc_roc=r["auc_roc"], auc_pr=r["auc_pr"], f1_theta1=r["analisis_theta1"]["f1"],
                               precision_theta1=r["analisis_theta1"]["precision"], recall_theta1=r["analisis_theta1"]["recall"],
                               fpr_theta1=r["analisis_theta1"]["fpr"], f1_theta2=r["analisis_theta2"]["f1"],
                               fpr_theta2=r["analisis_theta2"]["fpr"], alpha=r["calibracion"]["alpha"],
                               theta1=r["calibracion"]["theta1"], theta2=r["calibracion"]["theta2"]) for k, r in res.items()])
    tabla.to_csv(OUT / "tabla_comparativa_cert.csv", index=False)

    # Evaluación por unidad funcional ("PYME simulada": unidades de decenas a cientos de empleados)
    por_unidad = []
    col_u = "functional_unit" if "functional_unit" in te.columns else None
    if col_u:
        for u, idx in te.groupby(col_u).indices.items():
            y = te["anomalo"].to_numpy()[idx]
            m = aa.binary_metrics(y, (decs["hibrido"][idx] >= 1).astype(int))
            por_unidad.append(dict(unidad=u, empleados=int(te.iloc[idx]["user"].nunique()), dias=len(idx),
                                   dias_maliciosos=int(y.sum()), f1=m["f1"], recall=m["recall"], fpr=m["fpr"]))
    por_unidad = pd.DataFrame(por_unidad)
    if len(por_unidad):
        por_unidad.to_csv(OUT / "resultados_por_unidad_cert.csv", index=False)

    imp = {}
    rng = np.random.default_rng(cfg.seed)
    yt = te["anomalo"].to_numpy()
    if yt.sum() > 0:
        base = average_precision_score(yt, p_t)
        for c in F:
            Xp = te[F].to_numpy().copy(); j = F.index(c); Xp[:, j] = rng.permutation(Xp[:, j])
            imp[c] = round(float(base - average_precision_score(yt, hgb.predict_proba(Xp)[:, 1])), 4)
    resultados = dict(
        dataset="CERT Insider Threat r4.2 (CMU)", unidad="usuario-día (solo inicios/cierres de sesión)",
        etiqueta="día con al menos un inicio de sesión listado como malicioso en answers",
        particion="por empleado, estratificada (70/15/15)",
        volumen=dict(eventos=len(lg), empleados=int(lg["user"].nunique()), insiders=int(len(ins)),
                     usuario_dias=len(a), dias_maliciosos=int(a["anomalo"].sum()),
                     dias_cualquier_malicioso=int(a["anomalo_cualquier"].sum())),
        modelos=res, por_unidad=por_unidad.round(4).to_dict(orient="records"),
        prueba_pareada_hibrido_vs_reglas=prueba_pareada(te, decs["reglas_estaticas"], decs["hibrido"]),
        prueba_pareada_hibrido_vs_iforest=prueba_pareada(te, decs["iforest"], decs["hibrido"]),
        importancia_supervisado=dict(sorted(imp.items(), key=lambda kv: -kv[1])),
        latencia_ms=dict(iforest=aa.inference_latency_ms(iso, te, features=F),
                         supervisado=aa.inference_latency_ms(hgb, te, features=F)),
    )
    aa.to_json(resultados, OUT / "metricas_cert.json")
    figures.model_comparison(tabla.fillna(0), OUT / "figuras" / "comparacion_modelos_cert.png")
    if imp:
        figures.shap_bars(pd.Series(imp).clip(lower=0), OUT / "figuras" / "importancia_cert.png",
                          xlabel="Caída de AUC-PR al permutar la variable")
    print(tabla.round(3).to_string(index=False))
    if len(por_unidad):
        print("[por unidad funcional]"); print(por_unidad.round(3).to_string(index=False))
    for k in ("prueba_pareada_hibrido_vs_reglas", "prueba_pareada_hibrido_vs_iforest"):
        print(f"[{k}]", {kk: (round(v, 4) if isinstance(v, float) else v) for kk, v in resultados[k].items()
                         if kk not in ("f1_baseline", "f1_adaptauth")})
    print(f"[listo] {time.time() - t0:.0f}s -> {OUT}")


if __name__ == "__main__":
    main()


In [ ]:
import os, glob, json, pandas as pd
from IPython.display import Image, display
from google.colab import drive
drive.mount("/content/drive")
CARPETA = "/content/drive/MyDrive/AdaptAuth-ML"
os.makedirs(CARPETA, exist_ok=True)
TMP = "/content"

## Parte A. SuspiciousLogin (Google Workspace)

In [ ]:
if not os.path.exists(f"{TMP}/SuspiciousLoginDataset"):
    !git clone -q --depth 1 https://github.com/salomaopena/SuspiciousLoginDataset.git "{TMP}/SuspiciousLoginDataset"
SL = glob.glob(f"{TMP}/SuspiciousLoginDataset/**/suspicious_logins_public_v1.csv", recursive=True)[0]
!python run_suspiciouslogin.py --data "{SL}"
!rm -rf "{CARPETA}/results_sl" && cp -r results_sl "{CARPETA}/results_sl"

In [ ]:
display(pd.read_csv("results_sl/tabla_comparativa_sl.csv").round(4))
display(pd.read_csv("results_sl/puntos_operacion_sl.csv").round(4))
for f in ["comparacion_modelos_sl", "puntos_operacion_sl", "importancia_grupos_sl", "f1_por_grupo_sl"]:
    display(Image(f"results_sl/figuras/{f}.png"))

## Parte B. CERT r4.2: descarga (solo la primera vez) y extracción de logon.csv, LDAP y respuestas

In [ ]:
CERT = f"{CARPETA}/CERT_r4.2"
os.makedirs(CERT, exist_ok=True)
tiene_logon = glob.glob(f"{CERT}/**/logon.csv", recursive=True)
tiene_answers = glob.glob(f"{CERT}/**/insiders.csv", recursive=True)
if tiene_logon and tiene_answers:
    print("CERT r4.2 ya está en tu Drive:", tiene_logon[0])
else:
    !apt-get -qq install -y lbzip2 > /dev/null 2>&1
    if not tiene_answers:
        if not os.path.exists(f"{TMP}/answers.tar.bz2"):
            !wget -q --show-progress -O "{TMP}/answers.tar.bz2" "https://ndownloader.figshare.com/files/24857828"
        !tar -xjf "{TMP}/answers.tar.bz2" -C "{CERT}"
    if not tiene_logon:
        if not os.path.exists(f"{TMP}/r4.2.tar.bz2"):
            print("Descargando CERT r4.2 (4.8 GB, solo esta vez)...")
            !wget -q --show-progress -O "{TMP}/r4.2.tar.bz2" "https://ndownloader.figshare.com/files/24856766"
        print("Extrayendo solo logon.csv y LDAP (unos minutos)...")
        !mkdir -p "{TMP}/cert_tmp" && tar --use-compress-program=lbzip2 -xf "{TMP}/r4.2.tar.bz2" -C "{TMP}/cert_tmp" --wildcards "*logon.csv" "*LDAP/*"
        lp = glob.glob(f"{TMP}/cert_tmp/**/logon.csv", recursive=True)[0]
        ldap = os.path.join(os.path.dirname(lp), "LDAP")
        !cp "{lp}" "{CERT}/logon.csv" && cp -r "{ldap}" "{CERT}/LDAP"
        !rm -rf "{TMP}/cert_tmp" "{TMP}/r4.2.tar.bz2"
!ls -lh "{CERT}"

## Parte B. Experimento CERT (usuario-día)

In [ ]:
!python run_cert.py --dir "{CERT}"
!rm -rf "{CARPETA}/results_cert" && cp -r results_cert "{CARPETA}/results_cert"

In [ ]:
display(pd.read_csv("results_cert/tabla_comparativa_cert.csv").round(4))
if os.path.exists("results_cert/resultados_por_unidad_cert.csv"):
    display(pd.read_csv("results_cert/resultados_por_unidad_cert.csv").round(4))
for f in ["comparacion_modelos_cert", "importancia_cert"]:
    if os.path.exists(f"results_cert/figuras/{f}.png"):
        display(Image(f"results_cert/figuras/{f}.png"))
print("Listo. Mándame results_cert/metricas_cert.json desde tu Drive.")